In [3]:
import pandas as pd
from shareplum import Office365, Site
from shareplum.site import Version
import streamlit as st
from datetime import datetime
from shareplum import Site, Office365
from shareplum.site import Version
import json

USERNAME = "biosafety@blisshealthcare.co.ke"
PASSWORD = "#Safety2026"
SHAREPOINT_URL = "https://blissgvske.sharepoint.com"
SHAREPOINT_SITE = "https://blissgvske.sharepoint.com/sites/BlissHealthcareReports/opticalreports/"

class SharePoint:
    def auth(self):
        try:
            self.authcookie = Office365(
                SHAREPOINT_URL,
                username=USERNAME,
                password=PASSWORD
            ).GetCookies()

            self.site = Site(
                SHAREPOINT_SITE,
                version=Version.v365,
                authcookie=self.authcookie
            )
            return self.site

        except Exception as e:
            print(f"Authentication failed: {e}")
            raise
    
    def connect_to_list(self, ls_name, columns=None, query=None, next_page=None):
        try:
            self.auth_site = self.auth()
            sp_list = self.auth_site.List(list_name=ls_name)
            
            # Fetch list data with query and pagination if applicable
            if next_page:
                list_data = sp_list.GetListItems(query=query, next_page=next_page)
            else:
                list_data = sp_list.GetListItems(query=query)
            
            # If the list_data is a list, process it directly
            if isinstance(list_data, list):
                if columns:
                    filtered_list_data = [
                        {col: item.get(col, None) for col in columns}
                        for item in list_data
                    ]
                    next_page_url = None
                else:
                    filtered_list_data = list_data
                    next_page_url = None

                return {'results': filtered_list_data, '__next': next_page_url}
            else:
                raise ValueError("Unexpected data format returned from SharePoint")
        
        except Exception as e:
            raise e

@st.cache_data(ttl=80, max_entries=2000, show_spinner=False, persist=False, experimental_allow_widgets=False)
def load_new():
    # Define the columns to retrieve from SharePoint
    columns = [
        "DATE OF ORDER", "PATIENT NAME", "MRN OR MCC", "SPECIAL REMARKS", "PRODUCT TYPE", "BRANCH", "SIDE",
        "SIMPLECODE", "SEARCH NAME", "SPHERE", "CYLINDER", "AXIS", "ADDITION", "QTY", "HEIGHT", "PD",
        "SHAPE", "FRAME TYPE", "A", "DBL", "B", "SCHEME", "FRAME SELECTION TYPE", "VENDOR TYPE", 
        "NAME OF STAFF", "ORDER STATUS", "APPROVED STATUS", "LENS PRODUCT TYPE", "PHONE", "Created"
    ]
    
    # Define date range filter
    start_date = "2024-10-23T00:00:00Z"
    end_date = "2024-10-28T23:59:59Z"
    
    # SharePoint CAML query to filter the Created column
    query = {
        "Where": [
            {
                "And": [
                    {"Geq": {"FieldRef": "Created", "Value": start_date}},
                    {"Leq": {"FieldRef": "Created", "Value": end_date}}
                ]
            }
        ]
    }
    
    # Create an instance of the SharePoint class
    sp = SharePoint()
    
    # Connect to the SharePoint list and retrieve data
    list_data = sp.connect_to_list(ls_name="Optical 2022", columns=columns, query=query)
    
    # Convert the retrieved data to a DataFrame
    if 'results' in list_data:
        book_df = pd.DataFrame(list_data['results'])
    else:
        book_df = pd.DataFrame()  # Return an empty DataFrame if no data is retrieved

    return book_df

# Call load_new to retrieve and cache data
book_df = load_new()


2024-10-29 16:49:08.054 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Authentication failed: ('Error authenticating against Office 365. Error from Office 365:', 'AADSTS50126: Error validating credentials due to invalid username or password.')


Exception: ('Error authenticating against Office 365. Error from Office 365:', 'AADSTS50126: Error validating credentials due to invalid username or password.')